In [42]:
import tensorflow as tf
import numpy as np 
import matplotlib.pyplot as plt
import cv2 
import pandas as pd
from tensorflow import keras


In [43]:
file_path = './data/fer2013.csv'
df = pd.read_csv(file_path)

Extracting the image data form the .csv file


In [44]:
train_data = df[df['Usage'] == 'Training']
val_data = df[df['Usage'] == 'PublicTest']
test_data = df[df['Usage'] == 'PrivateTest']

#To generate a 48*48 image matrix 
def pixel_preprocess(df):
    pixel_series = df['pixels'].apply(lambda x: np.fromstring(x,sep=' ',dtype='float32'))
    x = np.vstack(pixel_series.values)
    x = x.reshape(-1,48,48,1)
    x = x / 255.0
    y = keras.utils.to_categorical(df['emotion'].values,num_classes=7)
    return x, y

X_train, y_train = pixel_preprocess(train_data)
X_val, y_val     = pixel_preprocess(val_data)
X_test, y_test   = pixel_preprocess(test_data)



0: Angry
1: Disgust
2: Fear
3: Happy
4: Sad
5: Surprise
6: Neutral

Visualize the Testing data extracted form the .csv file

In [ ]:
i = 0
for img, label in zip(X_train, y_train):  
    
    if label[6] == 1:
        i += 1
        plt.imshow(img.reshape(48, 48), cmap='gray')
        plt.axis('off')
        plt.show()
        if i >= 5:
            break
    

In [45]:
import pickle 
pickle_out = open('X_train.pickle','wb')
pickle.dump(X_train,pickle_out)
pickle_out = open('y_train.pickle','wb')
pickle.dump(y_train,pickle_out)

pickle_out = open('X_val.pickle','wb')
pickle.dump(X_val,pickle_out)
pickle_out = open('y_val.pickle','wb')
pickle.dump(y_val,pickle_out)

pickle_out = open('X_test.pickle','wb')
pickle.dump(X_test,pickle_out)
pickle_out = open('y_test.pickle','wb')
pickle.dump(y_test,pickle_out)


In [5]:
#CNN model 
import numpy as np
import tensorflow as tf
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
from keras.layers import BatchNormalization
import pickle

#Read all the previously dumped data
X_test = pickle.load(open("X_test.pickle", "rb"))
#y_test = pickle.load(open("y_test.pickle", "rb"))
X_train = pickle.load(open("X_train.pickle", "rb"))
y_train = pickle.load(open("y_train.pickle", "rb"))
X_val = pickle.load(open("X_val.pickle", "rb"))
y_val = pickle.load(open("y_val.pickle", "rb"))




In [6]:
model = Sequential()

# Block 1
model.add(Conv2D(64, (3,3), padding='same', input_shape=X_test.shape[1:]))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(Conv2D(64, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))


# Block 2
model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))


# Block 3
model.add(Conv2D(256, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation("relu"))

model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))


# Dense Layers
model.add(Flatten())

model.add(Dense(512))
model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(Dropout(0.5))

model.add(Dense(256))
model.add(BatchNormalization())
model.add(Activation("relu"))
model.add(Dropout(0.5))


# Output
model.add(Dense(7))
model.add(Activation('softmax'))


model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# 1. Define Callbacks (Safety Nets)
# Stops training if validation loss doesn't improve for 5 epochs
early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1, restore_best_weights=True)

# Lowers the learning rate if the model gets "stuck"
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1, min_lr=0.0001)

# Saves only the best version of your model to your E: drive
checkpoint = ModelCheckpoint('emotion_model_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1)

# 2. Run the Fit
history = model.fit(
    X_train, y_train,
    batch_size=64,             # How many images the GTX 1650 processes at once
    epochs=50,                 # Maximum number of passes through the data
    validation_data=(X_val, y_val), 
    #callbacks=[early_stopping, reduce_lr, checkpoint],
    shuffle=True               # Mixes the data every epoch to prevent patterns
)

Epoch 1/50
449/449 [==============================] - 30s 55ms/step - loss: 1.8061 - accuracy: 0.3255 - val_loss: 1.7221 - val_accuracy: 0.3558
Epoch 2/50
449/449 [==============================] - 23s 52ms/step - loss: 1.3876 - accuracy: 0.4664 - val_loss: 1.5136 - val_accuracy: 0.4085
Epoch 3/50
449/449 [==============================] - 23s 52ms/step - loss: 1.2567 - accuracy: 0.5203 - val_loss: 1.2711 - val_accuracy: 0.5124
Epoch 4/50
449/449 [==============================] - 23s 51ms/step - loss: 1.1799 - accuracy: 0.5464 - val_loss: 1.2384 - val_accuracy: 0.5138
Epoch 5/50
449/449 [==============================] - 24s 54ms/step - loss: 1.1247 - accuracy: 0.5727 - val_loss: 1.1539 - val_accuracy: 0.5561
Epoch 6/50
449/449 [==============================] - 24s 53ms/step - loss: 1.0761 - accuracy: 0.5935 - val_loss: 1.1072 - val_accuracy: 0.5734
Epoch 7/50
449/449 [==============================] - 24s 53ms/step - loss: 1.0381 - accuracy: 0.6068 - val_loss: 1.0754 - val_accuracy:

In [11]:
model.save("emotion_detector")

INFO:tensorflow:Assets written to: emotion_detector\assets


In [12]:
import cv2 
import numpy as np 
import tensorflow as tf

from tensorflow.python.keras.models import load_model
model = load_model("emotion_detector")
emotion_labels = [
    "Angry", 
    "Disgust", 
    "Fear", 
    "Happy", 
    "Sad", 
    "Surprise", 
    "Neutral"
]
# Load face detector (Haar cascade)
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]

        # Resize to 48x48 (same as training)
        face = cv2.resize(face, (48, 48))

        # Normalize like training
        face = face / 255.0

        # Reshape to model input
        face = np.reshape(face, (1, 48, 48, 1))

        # Predict
        predictions = model.predict(face, verbose=0)
        emotion = emotion_labels[np.argmax(predictions)]

        # Draw rectangle + label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, emotion, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9, (0,255,0), 2)

    cv2.imshow("Emotion Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
